# ETF Explorer (v1)

Discovery view over Stashaway's full ETF Explorer offering (~98 ETFs).
Computes multi-window metrics (1Y/3Y/5Y) + correlation with your combined book.
Renders to `reports/etf_explorer.html`.

In [1]:
from datetime import date
from pathlib import Path

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.etf_explorer import build_etf_explorer, render_etf_explorer_report
from hailmary.allocation.portfolios import from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.allocation.returns import last_business_day_on_or_before
from hailmary.data.providers import YahooFinanceProvider

ETF_XLSX = Path('../../data/stashaway_etf_universe.xlsx')
STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
REPORT_PATH = Path('../../reports/etf_explorer.html')
START = date(2020, 1, 1)
END = last_business_day_on_or_before(date.today())
TARGET_ANN_RETURN = 0.05
print(f'window: {START}..{END}')

window: 2020-01-01..2026-05-25


## Load user's book (for correlation reference)

In [2]:
parsed = parse_statement(STATEMENT_PATH)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
provider = YahooFinanceProvider()
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
print(f'{len(portfolios)} portfolios loaded')

2026-05-25 00:29:37.138 | DEBUG    | hailmary.allocation.statements:get:169 - Statement cache hit for 2026-04 StashAway Monthly Statement.pdf


2026-05-25 00:29:37.138 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=fa570043d3f4


15 portfolios loaded


## Build the explorer DataFrame

In [3]:
explorer_df = build_etf_explorer(
    ETF_XLSX,
    portfolios=portfolios,
    price_source=provider,
    fx_series_usd_sgd=fx_series_usd_sgd,
    start=START,
    end=END,
)
print(f'{len(explorer_df)} ETFs, {explorer_df["has_data"].sum()} with Yahoo data')
explorer_df.head(20)

2026-05-25 00:29:37.437 | INFO     | hailmary.allocation.etf_explorer:build_etf_explorer:194 - ETF Explorer: fetching 97 symbols from Yahoo…


2026-05-25 00:29:37.437 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=ead95bfc3ee5


2026-05-25 00:29:37.563 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=333aa9072c3e


2026-05-25 00:29:37.586 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=8ca47a4035ad


2026-05-25 00:29:37.586 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=b328b58cfa0a


...

2026-05-25 00:29:37.697 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=da20efc4ae97


2026-05-25 00:29:37.780 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=b328b58cfa0a


2026-05-25 00:29:37.799 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=e443f1c4bd4f


98 ETFs, 98 with Yahoo data


,asset_class,name,ticker,wrapper,fund_manager,has_data,n_days,cum_return_1M,cum_return_3M,ytd_return,...,ann_return_5Y,max_dd_5Y,vol_5Y,corr_n,corr_book_1M,corr_book_3M,corr_book_YTD,corr_book_1Y,corr_book_3Y,corr_book_5Y
0,All Country World,iShares MSCI ACWI UCITS ETF,ISAC.L,UCITS (LSE),iShares,True,1577,4.835825e-02,0.084831,0.091182,...,0.114440,-0.218839,0.155481,283,0.876436,0.746963,0.724392,0.506175,0.515282,0.515282
1,Artificial Intelligence,Xtrackers Artificial Intelligence & Big Data U...,XAID.L,?,Xtrackers,True,1577,1.382875e-01,0.258760,0.197236,...,0.135270,-0.353230,0.222392,283,0.634796,0.585226,0.544790,0.395476,0.408078,0.408078
2,Asia ex-Japan,iShares MSCI All Country Asia ex Japan ETF,AAXJ,US,iShares,True,1561,7.245863e-02,0.100462,0.239316,...,0.102130,-0.347962,0.202936,283,0.853035,0.748472,0.740159,0.753000,0.743373,0.743373
3,Asia High Yield USD Corporate Bonds *,iShares USD Asia High Yield Bond ETF,QL3.SI,SG,iShares,True,1562,-4.440892e-16,0.013327,0.020372,...,-0.025536,-0.414487,0.092206,277,-0.398751,0.027238,0.037684,0.091311,0.101859,0.101859
4,Australia,iShares MSCI Australia ETF,EWA,US,iShares,True,1561,-1.519149e-02,-0.012245,0.101376,...,0.102646,-0.203303,0.203792,283,0.841924,0.721020,0.713187,0.770951,0.762091,0.762091
5,Aerospace & Defense,Invesco Aerospace & Defense ETF,PPA,US,Invesco,True,1561,1.089840e-02,-0.038184,0.101050,...,0.213403,-0.173779,0.189379,283,0.750084,0.706770,0.700283,0.748377,0.728856,0.728856
6,Battery Value-chain,L&G Battery Value-Chain UCITS ETF,BATT.L,?,LGIM,True,1577,2.941730e-02,0.221943,0.323620,...,0.140503,-0.367230,0.251855,283,0.609481,0.608617,0.601628,0.444530,0.431827,0.431827
7,Biotechnology,iShares Biotechnology ETF,IBB,US,iShares,True,1561,-1.442351e-02,-0.024962,-0.015069,...,0.041731,-0.350260,0.221932,283,0.742861,0.771489,0.687708,0.610805,0.594474,0.594474
8,Bitcoin (Accredited Investors only),Fidelity® Wise Origin® Bitcoin Fund,FBTC,?,Fidelity,True,575,-2.343784e-02,0.188122,-0.065190,...,0.305041,-0.468593,0.510024,283,0.720320,0.839045,0.825998,0.709713,0.706869,0.706869
9,Blockchain,Invesco CoinShares Global Blockchain UCITS ETF,BCHN.L,UCITS (LSE),Invesco,True,1575,1.035695e-01,0.241038,0.189784,...,0.050032,-0.573107,0.382672,283,0.834082,0.686955,0.667032,0.416687,0.424759,0.424759


## Render HTML report

In [4]:
out = render_etf_explorer_report(
    explorer_df,
    REPORT_PATH,
    target_ann_return=TARGET_ANN_RETURN,
)
print(f'Wrote {out.resolve()}')

Wrote C:\Users\Dalva\src\project-hail-mary\reports\etf_explorer.html
